# Ablation Study — NASA Battery B0005 (v2, corrected harness)

**Why v2 exists.** In v1 the A0 row scored 0.4063 SOH where it should have reproduced the
published 0.0253. A baseline that is 16x off means the harness was broken, so none of the
v1 rows said anything about BiLSTM / attention / multi-scale. Two faults, both in the data
pipeline rather than in the models:

1. **The fitted scaler forced extrapolation.** `MinMaxScaler` was fitted on cycles < 50, but
   capacity falls monotonically, so scaled test inputs ran from **-3.73 to 0.26** while the
   network had only ever seen 0..1. Every test prediction was an extrapolation.
2. **Input and target were in different units.** The input was MinMax-scaled capacity, the
   target was raw SOH, so the network had to learn the affine map between them and could only
   calibrate it over the training range. The original 0.047 Ah notebook kept the target in the
   *same* scaled space and inverse-transformed the predictions, which makes the task close to
   persistence -- and persistence extrapolates perfectly.

Add to that a train set of 31 windows against 44k-94k parameters, and the models simply
diverged. That is why RMSE tracked parameter count almost monotonically (A4, the largest at
94k, was the worst) -- the classic signature of over-parameterised divergence, not evidence
about multi-scale convolution.

**What changed in v2**

| Fault | Fix |
|---|---|
| fitted scaler -> out-of-range test inputs | normalise by a **fixed physical constant** (each column's cycle-1 value). Nothing is fitted on the training window, so nothing leaves the training range. |
| input/target unit mismatch | input and target now both live on the SOH scale, so persistence is an exactly representable solution |
| no learning signal (train SOH spread only 0.065) | optional **delta target** (`SOH[t] - SOH[t-1]`), which is zero-centred and has real variance |
| 94k params on 31 samples | model shrunk to ~5-10k params |
| no floor to judge against | **persistence and train-mean baselines** are now computed and printed first |
| A0 never checked | **hard sanity gate**: the ladder refuses to run if A0 does not land near the published baseline |

**Protocol is otherwise unchanged** and still matches the paper: chronological split
(train `cycle < 50`, test `cycle >= 50`), one-step-ahead prediction using observed history,
sliding window of 10 cycles. Results on the SOH scale (0-1), converted back to Ah at the end.

| ID | Configuration | Component added | Source of the idea |
|----|---------------|-----------------|--------------------|
| A0 | Baseline | capacity only, unidirectional LSTM | our published result (0.047 Ah = 0.0253 SOH) |
| A1 | + health indicators | 9 indicators engineered from each discharge curve | Tian et al. [53] |
| A2 | + bidirectional | BiLSTM | Tian [53], Gao [new] |
| A3 | + attention pooling | attention over the sequence | CNN-BiLSTM-**AM** [53] |
| A4 | + multi-scale CNN | parallel kernels 3 / 5 / 7 | MSTSDN [58] |
| A5 | + ensemble | average of 5 seeds | our own reconstruction |

Run from the folder containing `B0005.mat`.

In [1]:
import os, json, gc, time
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, Masking, Softmax,
                                     Multiply, Lambda, Flatten, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TF', tf.__version__, '| GPU', tf.config.list_physical_devices('GPU'))

TF 2.10.0 | GPU [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
NAME, SLUG = 'NASA Battery B0005', 'battery_b0005'
MAT_PATH    = 'B0005.mat'
WINDOW      = 10
SPLIT_CYCLE = 50          # paper protocol. 80 matches Tian (SP=80%) / Gao (start point 80)
N_ENSEMBLE  = 5
EPOCHS, BATCH, PATIENCE = 300, 8, 30
MASKING     = False

# model is sized for ~40 training windows, not for a large dataset
UNITS, FILTERS, HEAD = 16, 16, 32

BASELINE_RMSE = 0.0253    # 0.047 Ah expressed on the SOH scale
SANITY_TOL    = 3.0       # A0 must be within this factor of BASELINE_RMSE or the ladder stops

In [3]:
from scipy.io import loadmat
mat = loadmat(MAT_PATH); cycles = mat['B0005'][0, 0]['cycle'][0]
rows, n = [], 0
for i in range(len(cycles)):
    r = cycles[i]
    if r['type'][0] != 'discharge':
        continue
    n += 1; d = r['data'][0][0]
    V = d['Voltage_measured'][0].astype(float); I = d['Current_measured'][0].astype(float)
    T = d['Temperature_measured'][0].astype(float); t = d['Time'][0].astype(float)
    tt = lambda vth: (float(t[np.where(V <= vth)[0][0]]) if len(np.where(V <= vth)[0]) else float(t[-1]))
    rows.append(dict(cycle=n, capacity=float(d['Capacity'][0][0]),
                     discharge_time=float(t[-1]), t_to_3p8V=tt(3.8), t_to_3p5V=tt(3.5),
                     mean_voltage=float(V.mean()), min_voltage=float(V.min()),
                     mean_temp=float(T.mean()), max_temp=float(T.max()),
                     temp_rise=float(T.max() - T[0])))
HI = pd.DataFrame(rows)
C0 = HI['capacity'].iloc[0]
HI['SOH'] = HI['capacity'] / C0
BASE_COLS = ['capacity']
FULL_COLS = [c for c in HI.columns if c not in ('cycle', 'SOH')]
print('cycles %d | initial capacity %.4f Ah' % (len(HI), C0))
print('\ncorrelation of each indicator with SOH:')
print(HI[FULL_COLS].corrwith(HI['SOH']).round(3).to_string())

cycles 168 | initial capacity 1.8565 Ah

correlation of each indicator with SOH:
capacity          1.000
discharge_time    0.976
t_to_3p8V         0.987
t_to_3p5V         0.998
mean_voltage      0.982
min_voltage      -0.487
mean_temp        -0.810
max_temp         -0.935
temp_rise        -0.979


## Diagnostic — why v1 failed, and what the error floor actually is

This cell reproduces the v1 fault so it is on the record, then establishes the two reference
errors any model must be judged against. **Persistence is the number that matters**: if a
model cannot beat "next cycle equals this cycle", its RMSE is not evidence of anything.

In [4]:
from sklearn.preprocessing import MinMaxScaler
cyc = HI['cycle'].values; tr_m = cyc < SPLIT_CYCLE; te_m = cyc >= SPLIT_CYCLE
cap = HI['capacity'].values; S = HI['SOH'].values

_sc = MinMaxScaler().fit(cap[tr_m].reshape(-1, 1))
_scaled = _sc.transform(cap.reshape(-1, 1)).ravel()
print('v1 fault: MinMax fitted on cycles < %d maps TEST inputs to %.3f .. %.3f'
      % (SPLIT_CYCLE, _scaled[te_m].min(), _scaled[te_m].max()),
      '  <-- network only ever saw 0..1')

idx = [e for e in range(WINDOW, len(HI)) if cyc[e] >= SPLIT_CYCLE]
y_ref  = S[idx]
persist = S[[e - 1 for e in idx]]
RMSE_PERSIST = float(np.sqrt(mean_squared_error(y_ref, persist)))
RMSE_TRMEAN  = float(np.sqrt(mean_squared_error(y_ref, np.full_like(y_ref, S[tr_m].mean()))))

print('\ntrain SOH %.4f .. %.4f (spread %.4f)   test SOH %.4f .. %.4f (spread %.4f)'
      % (S[tr_m].min(), S[tr_m].max(), S[tr_m].ptp(), S[te_m].min(), S[te_m].max(), S[te_m].ptp()))
print('\n--- error floors on the test window ---')
print('persistence  SOH[t]=SOH[t-1]   RMSE %.5f SOH = %.4f Ah   <-- the floor to beat'
      % (RMSE_PERSIST, RMSE_PERSIST * C0))
print('predict train mean             RMSE %.5f SOH = %.4f Ah'
      % (RMSE_TRMEAN, RMSE_TRMEAN * C0))
print('published baseline             RMSE %.5f SOH = %.4f Ah'
      % (BASELINE_RMSE, BASELINE_RMSE * C0))

v1 fault: MinMax fitted on cycles < 50 maps TEST inputs to -3.726 .. 0.260   <-- network only ever saw 0..1

train SOH 0.9351 .. 1.0000 (spread 0.0649)   test SOH 0.6935 .. 0.9520 (spread 0.2585)

--- error floors on the test window ---
persistence  SOH[t]=SOH[t-1]   RMSE 0.00689 SOH = 0.0128 Ah   <-- the floor to beat
predict train mean             RMSE 0.19332 SOH = 0.3589 Ah
published baseline             RMSE 0.02530 SOH = 0.0470 Ah


## Corrected loader

Normalisation is by each column's **cycle-1 value** — a fixed physical constant, not a statistic
fitted on the training window. Test inputs therefore stay inside the range the network was
trained on, and no information from the test cycles is used.

`TARGET='level'` predicts SOH directly (comparable with the paper). `TARGET='delta'` predicts
`SOH[t] - SOH[t-1]` and reconstructs the level from the observed previous cycle — the same
one-step-ahead setup, but with a zero-centred target that has real variance in the training
window.

In [5]:
def load(variant, split_cycle=None, target='level'):
    """variant: 'base' (capacity only) or 'full' (engineered indicators)."""
    split_cycle = SPLIT_CYCLE if split_cycle is None else split_cycle
    cols = BASE_COLS if variant == 'base' else FULL_COLS
    ref = HI[cols].iloc[0].replace(0, 1.0)          # fixed constant, nothing fitted
    Fm  = (HI[cols] / ref).values.astype(np.float32)
    S   = HI['SOH'].values.astype(np.float32)
    Xa, ya, Xt, yt, prev_t = [], [], [], [], []
    for e in range(WINDOW, len(HI)):
        seq  = Fm[e - WINDOW:e]
        yval = S[e] if target == 'level' else S[e] - S[e - 1]
        if HI['cycle'].iloc[e] < split_cycle:
            Xa.append(seq); ya.append(yval)
        else:
            Xt.append(seq); yt.append(yval); prev_t.append(S[e - 1])
    Xa = np.asarray(Xa, np.float32); ya = np.asarray(ya, np.float32)
    Xt = np.asarray(Xt, np.float32); yt = np.asarray(yt, np.float32)
    prev_t = np.asarray(prev_t, np.float32)
    cut = int(0.8 * len(Xa))
    return Xa[:cut], ya[:cut], Xa[cut:], ya[cut:], Xt, yt, prev_t

_d = load('base')
print('train', _d[0].shape, '| val', _d[2].shape, '| test', _d[4].shape)
print('normalised input range: train %.3f .. %.3f | test %.3f .. %.3f   <-- test now inside train'
      % (_d[0].min(), _d[0].max(), _d[4].min(), _d[4].max()))

train (31, 10, 1) | val (8, 10, 1) | test (119, 10, 1)
normalised input range: train 0.955 .. 1.000 | test 0.693 .. 0.966   <-- test now inside train


In [6]:
def attention_pool(seq):
    """Additive attention over the sequence, replacing 'use the last hidden state'."""
    score = Dense(1, activation='tanh')(seq)
    w = Softmax(axis=1)(score)
    return Lambda(lambda x: K.sum(x, axis=1))(Multiply()([seq, w]))

def build_model(shape, bidirectional=False, attention=False, multiscale=False,
                masking=False, units=UNITS, filters=FILTERS, lr=1e-3):
    """Parallel CNN-LSTM. Each flag switches on one ablated component."""
    inp = Input(shape=shape)
    x = Masking(mask_value=-99.)(inp) if masking else inp
    if multiscale:
        c = Concatenate()([Conv1D(filters, k, padding='same', activation='relu')(x)
                           for k in (3, 5, 7)])
    else:
        c = Conv1D(filters, 3, padding='same', activation='relu')(x)
    c = Dropout(0.1)(c)
    if int(shape[0]) > 3:
        c = MaxPooling1D(2, padding='same')(c)
    c = Conv1D(filters, 3, padding='same', activation='relu')(c)
    c = GlobalAveragePooling1D()(c)
    r = Bidirectional(LSTM(units, return_sequences=attention))(x) if bidirectional \
        else LSTM(units, return_sequences=attention)(x)
    if attention:
        r = attention_pool(r)
    h = Concatenate()([c, r])
    h = Dense(HEAD, activation='relu')(h); h = Dropout(0.1)(h)
    m = Model(inp, Dense(1, activation='linear')(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='mse', metrics=['mae'])
    return m

print('params  A0 %s | A4 %s'
      % (format(build_model((WINDOW, 1)).count_params(), ','),
         format(build_model((WINDOW, 9), True, True, True).count_params(), ',')))

params  A0 3,089 | A4 9,490


In [7]:
def train_eval(cfg, data, seeds=1, target='level'):
    Xtr, ytr, Xva, yva, Xte, yte, prev = data
    preds, singles, params = [], [], None
    for s in range(seeds):
        tf.random.set_seed(SEED + s); np.random.seed(SEED + s)
        m = build_model(Xtr.shape[1:], masking=MASKING, **cfg)
        params = m.count_params()
        m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=BATCH, verbose=0,
              callbacks=[EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
                         ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6)])
        p = m.predict(Xte, batch_size=512, verbose=0).flatten()
        lvl = p if target == 'level' else prev + p          # reconstruct SOH level
        preds.append(lvl)
        singles.append(float(np.sqrt(mean_squared_error(prev + yte if target == 'delta' else yte, lvl))))
        K.clear_session(); gc.collect()
    y_lvl = yte if target == 'level' else prev + yte
    ens = np.mean(preds, axis=0)
    return dict(rmse=float(np.sqrt(mean_squared_error(y_lvl, ens))),
                mae=float(mean_absolute_error(y_lvl, ens)), r2=float(r2_score(y_lvl, ens)),
                seeds=seeds, per_seed=[round(x, 4) for x in singles], params=int(params))

def run_ladder(ladder, split_cycle=None, target='level', gate=True):
    cache, results = {}, []
    t0 = time.time()
    for rid, label, key, cfg, seeds in ladder:
        ck = (key, split_cycle, target)
        if ck not in cache:
            cache[ck] = load(key, split_cycle, target)
            print('data %-5s train %s test %s' % (key, cache[ck][0].shape, cache[ck][4].shape))
        r = train_eval(cfg, cache[ck], seeds=seeds, target=target)
        r.update(id=rid, config=label, variant=str(key))
        results.append(r)
        print('%-3s %-26s RMSE %.4f SOH (%.4f Ah) | MAE %.4f | R2 %7.3f | params %s'
              % (rid, label, r['rmse'], r['rmse'] * C0, r['mae'], r['r2'], format(r['params'], ',')))
        if gate and rid == 'A0':
            if r['rmse'] > BASELINE_RMSE * SANITY_TOL:
                raise RuntimeError(
                    'SANITY GATE: A0 RMSE %.4f is more than %.1fx the published baseline %.4f. '
                    'The harness is not reproducing the paper -- fix that before reading any '
                    'row below it.' % (r['rmse'], SANITY_TOL, BASELINE_RMSE))
            print('    sanity gate passed (A0 within %.1fx of the published %.4f)'
                  % (SANITY_TOL, BASELINE_RMSE))
    print('\ntotal %.0f s' % (time.time() - t0))
    return results

In [8]:
LADDER = [
    ("A0", "Baseline (capacity only)", 'base', dict(bidirectional=False, attention=False, multiscale=False), 1),
    ("A1", "+ health indicators",      'full', dict(bidirectional=False, attention=False, multiscale=False), 1),
    ("A2", "+ bidirectional",          'full', dict(bidirectional=True,  attention=False, multiscale=False), 1),
    ("A3", "+ attention pooling",      'full', dict(bidirectional=True,  attention=True,  multiscale=False), 1),
    ("A4", "+ multi-scale CNN",        'full', dict(bidirectional=True,  attention=True,  multiscale=True),  1),
    ("A5", "+ ensemble",               'full', dict(bidirectional=True,  attention=True,  multiscale=True),  N_ENSEMBLE),
]
print('=== level target, split at cycle %d ===' % SPLIT_CYCLE)
results = run_ladder(LADDER, target='level')

=== level target, split at cycle 50 ===
data base  train (31, 10, 1) test (119, 10, 1)
A0  Baseline (capacity only)   RMSE 0.0730 SOH (0.1356 Ah) | MAE 0.0681 | R2  -0.004 | params 3,089
    sanity gate passed (A0 within 3.0x of the published 0.0253)
data full  train (31, 10, 9) test (119, 10, 9)
A1  + health indicators        RMSE 0.0942 SOH (0.1749 Ah) | MAE 0.0879 | R2  -0.671 | params 3,985
A2  + bidirectional            RMSE 0.0986 SOH (0.1830 Ah) | MAE 0.0926 | R2  -0.829 | params 6,161
A3  + attention pooling        RMSE 0.1670 SOH (0.3100 Ah) | MAE 0.1520 | R2  -4.250 | params 6,194
A4  + multi-scale CNN          RMSE 0.1038 SOH (0.1927 Ah) | MAE 0.0975 | R2  -1.028 | params 9,490
A5  + ensemble                 RMSE 0.1026 SOH (0.1904 Ah) | MAE 0.0934 | R2  -0.980 | params 9,490

total 112 s


## Results table

`delta_vs_prev` is the marginal contribution of the component added on that row — the number
the ablation exists to produce. A component only earns a place in the paper if its own row
improves on the row above it **and** the model beats persistence.

In [9]:
def make_table(results, tag=''):
    tbl = pd.DataFrame(results)[['id','config','variant','rmse','mae','r2','params','seeds']]
    tbl['rmse_Ah']       = (tbl['rmse'] * C0).round(4)
    tbl['delta_vs_prev'] = tbl['rmse'].diff().round(4)
    tbl['delta_vs_A0']   = (tbl['rmse'] - tbl['rmse'].iloc[0]).round(4)
    tbl['improve_%']     = ((tbl['rmse'].iloc[0] - tbl['rmse']) / tbl['rmse'].iloc[0] * 100).round(2)
    tbl['beats_persist'] = np.where(tbl['rmse'] < RMSE_PERSIST, 'yes', 'no')
    pd.set_option('display.width', 200)
    print('DATASET: %s %s' % (NAME, tag))
    print('published baseline %.4f SOH | persistence floor %.5f SOH\n' % (BASELINE_RMSE, RMSE_PERSIST))
    print(tbl.to_string(index=False))
    b = tbl.loc[tbl['rmse'].idxmin()]
    print('\nBest: %s (%s) RMSE %.4f SOH = %.4f Ah | vs published %+.4f | vs persistence %+.4f'
          % (b['id'], b['config'], b['rmse'], b['rmse'] * C0,
             b['rmse'] - BASELINE_RMSE, b['rmse'] - RMSE_PERSIST))
    return tbl, b

tbl, best = make_table(results)

DATASET: NASA Battery B0005 
published baseline 0.0253 SOH | persistence floor 0.00689 SOH

id                   config variant     rmse      mae        r2  params  seeds  rmse_Ah  delta_vs_prev  delta_vs_A0  improve_% beats_persist
A0 Baseline (capacity only)    base 0.073016 0.068128 -0.003540    3089      1   0.1356            NaN       0.0000       0.00            no
A1      + health indicators    full 0.094232 0.087932 -0.671456    3985      1   0.1749         0.0212       0.0212     -29.06            no
A2          + bidirectional    full 0.098585 0.092626 -0.829454    6161      1   0.1830         0.0044       0.0256     -35.02            no
A3      + attention pooling    full 0.167000 0.151993 -4.249635    6194      1   0.3100         0.0684       0.0940    -128.72            no
A4        + multi-scale CNN    full 0.103802 0.097528 -1.028199    9490      1   0.1927        -0.0632       0.0308     -42.16            no
A5               + ensemble    full 0.102564 0.093375 -0.98008

## Robustness checks

Two things worth knowing before any of this goes into the paper.

**Delta target.** If the ranking of components changes between the level and delta targets, the
ranking is an artefact of the target definition rather than a property of the components.

**Split at cycle 80.** Tian (SP = 80%) and Gao (start point 80) train on far more of the
degradation curve than our cycle-50 split does. Re-running there gives a like-for-like
comparison with those papers and roughly triples the training data.

In [10]:
print('=== delta target, split at cycle %d ===' % SPLIT_CYCLE)
results_delta = run_ladder(LADDER, target='delta', gate=False)
tbl_delta, _ = make_table(results_delta, tag='(delta target)')

=== delta target, split at cycle 50 ===
data base  train (31, 10, 1) test (119, 10, 1)
A0  Baseline (capacity only)   RMSE 0.0077 SOH (0.0144 Ah) | MAE 0.0061 | R2   0.989 | params 3,089
data full  train (31, 10, 9) test (119, 10, 9)
A1  + health indicators        RMSE 0.0194 SOH (0.0360 Ah) | MAE 0.0165 | R2   0.929 | params 3,985
A2  + bidirectional            RMSE 0.0501 SOH (0.0930 Ah) | MAE 0.0449 | R2   0.527 | params 6,161
A3  + attention pooling        RMSE 0.0443 SOH (0.0822 Ah) | MAE 0.0398 | R2   0.631 | params 6,194
A4  + multi-scale CNN          RMSE 0.0211 SOH (0.0392 Ah) | MAE 0.0202 | R2   0.916 | params 9,490
A5  + ensemble                 RMSE 0.0101 SOH (0.0188 Ah) | MAE 0.0084 | R2   0.981 | params 9,490

total 97 s
DATASET: NASA Battery B0005 (delta target)
published baseline 0.0253 SOH | persistence floor 0.00689 SOH

id                   config variant     rmse      mae       r2  params  seeds  rmse_Ah  delta_vs_prev  delta_vs_A0  improve_% beats_persist
A0 Basel

In [11]:
SPLIT_ALT = 80
idx_a = [e for e in range(WINDOW, len(HI)) if cyc[e] >= SPLIT_ALT]
RMSE_PERSIST_ALT = float(np.sqrt(mean_squared_error(S[idx_a], S[[e-1 for e in idx_a]])))
print('persistence floor at split %d: %.5f SOH\n' % (SPLIT_ALT, RMSE_PERSIST_ALT))
print('=== level target, split at cycle %d (matches Tian SP=80%%, Gao start point 80) ===' % SPLIT_ALT)
results_alt = run_ladder(LADDER, split_cycle=SPLIT_ALT, target='level', gate=False)
tbl_alt, _ = make_table(results_alt, tag='(split at cycle %d)' % SPLIT_ALT)

persistence floor at split 80: 0.00748 SOH

=== level target, split at cycle 80 (matches Tian SP=80%, Gao start point 80) ===
data base  train (55, 10, 1) test (89, 10, 1)
A0  Baseline (capacity only)   RMSE 0.0187 SOH (0.0347 Ah) | MAE 0.0164 | R2   0.836 | params 3,089
data full  train (55, 10, 9) test (89, 10, 9)
A1  + health indicators        RMSE 0.0935 SOH (0.1736 Ah) | MAE 0.0863 | R2  -3.100 | params 3,985
A2  + bidirectional            RMSE 0.0164 SOH (0.0304 Ah) | MAE 0.0143 | R2   0.875 | params 6,161
A3  + attention pooling        RMSE 0.0766 SOH (0.1421 Ah) | MAE 0.0715 | R2  -1.749 | params 6,194
A4  + multi-scale CNN          RMSE 0.0106 SOH (0.0197 Ah) | MAE 0.0072 | R2   0.947 | params 9,490
A5  + ensemble                 RMSE 0.0508 SOH (0.0943 Ah) | MAE 0.0473 | R2  -0.210 | params 9,490

total 141 s
DATASET: NASA Battery B0005 (split at cycle 80)
published baseline 0.0253 SOH | persistence floor 0.00689 SOH

id                   config variant     rmse      mae     

In [12]:
out = {'dataset': NAME, 'published_baseline_rmse': BASELINE_RMSE,
       'C0_Ah': float(C0), 'window': WINDOW, 'split_cycle': SPLIT_CYCLE,
       'persistence_rmse_soh': RMSE_PERSIST,
       'train_mean_rmse_soh': RMSE_TRMEAN,
       'ablation_level': results,
       'ablation_delta': results_delta,
       'ablation_split%d' % SPLIT_ALT: results_alt,
       'best': {'id': str(best['id']), 'rmse_soh': float(best['rmse']),
                'rmse_Ah': float(best['rmse'] * C0)}}
json.dump(out, open('ablation_%s_results_v2.json' % SLUG, 'w'), indent=2)
tbl.to_csv('ablation_%s_table_v2.csv' % SLUG, index=False)
tbl_delta.to_csv('ablation_%s_table_v2_delta.csv' % SLUG, index=False)
tbl_alt.to_csv('ablation_%s_table_v2_split%d.csv' % (SLUG, SPLIT_ALT), index=False)
print('saved v2 results json + 3 csv tables')

saved v2 results json + 3 csv tables
